# Session 3: Model Evaluation and Comparison (Student Notebook)

In this session, you will learn how to systematically evaluate and compare AI model outputs using structured rubrics, automated judging, benchmarking, cost analysis, and A/B testing frameworks.

## Learning Objectives

By the end of this session, you will be able to:

1. Define and implement structured evaluation rubrics for AI responses
2. Use LLM-as-a-Judge for automated quality assessment
3. Benchmark response quality across different model configurations
4. Analyze latency and cost trade-offs between models
5. Design and run A/B tests to compare model configurations
6. Build end-to-end evaluation pipelines with reporting and visualization

In [ ]:
import openai
import json
import os
import time
import pandas as pd
import matplotlib.pyplot as plt

---
## Demo 1: Setting Up Evaluation Metrics and Scoring Rubrics

Before comparing models, we need a consistent framework for measuring quality. This demo shows how to define evaluation criteria and build a structured scoring rubric.

In [ ]:
client = openai.OpenAI()

# Define evaluation criteria for agentic responses
evaluation_criteria = {
    "relevance": "How relevant is the response to the question? (1-5)",
    "accuracy": "How factually accurate is the response? (1-5)",
    "completeness": "How complete and thorough is the response? (1-5)",
    "clarity": "How clear and well-structured is the response? (1-5)",
    "actionability": "How actionable are the suggestions/steps provided? (1-5)"
}

def create_scoring_rubric():
    rubric = {}
    for criterion, description in evaluation_criteria.items():
        rubric[criterion] = {
            "description": description,
            "1": "Poor - Fails to meet basic expectations",
            "2": "Below Average - Partially addresses the criterion",
            "3": "Average - Adequately meets the criterion",
            "4": "Good - Exceeds basic expectations",
            "5": "Excellent - Exceptional quality"
        }
    return rubric

rubric = create_scoring_rubric()
for criterion, details in rubric.items():
    print(f"\n{criterion.upper()}: {details['description']}")
    for score, desc in details.items():
        if score != "description":
            print(f"  {score}: {desc}")

---
## Demo 2: Automated Evaluation with LLM-as-a-Judge

Instead of manually scoring every response, we can use an LLM to act as a judge. This demo implements an automated evaluation function that scores responses against our rubric.

In [ ]:
def llm_judge(question, response_text, criteria):
    judge_prompt = f"""You are an expert evaluator. Rate the following AI response on a scale of 1-5 for each criterion.

Question asked: {question}

Response to evaluate:
{response_text}

Criteria to evaluate:
{json.dumps(criteria, indent=2)}

Return your evaluation as a JSON object with each criterion as a key and an object containing "score" (1-5) and "reasoning" (brief explanation) as the value."""

    client = openai.OpenAI()
    evaluation = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": judge_prompt}],
        response_format={"type": "json_object"},
        max_tokens=500
    )
    return json.loads(evaluation.choices[0].message.content)

# Test it
test_question = "How do I implement error handling in a Python web API?"
test_response = "Use try-except blocks to catch exceptions. Return appropriate HTTP status codes like 400 for bad requests and 500 for server errors. Log errors for debugging."

result = llm_judge(test_question, test_response, evaluation_criteria)
print(json.dumps(result, indent=2))

---
## Demo 3: Benchmarking Response Quality Across Models

Now we benchmark multiple model configurations against the same set of prompts, collecting metrics like response length, latency, and token usage.

In [ ]:
# Benchmark different configurations
test_prompts = [
    "Explain the concept of dependency injection in software engineering.",
    "What are the best practices for securing a REST API?",
    "How do you design a scalable microservices architecture?"
]

configurations = [
    {"model": "gpt-4o-mini", "temperature": 0.0, "label": "GPT-4o-mini (T=0)"},
    {"model": "gpt-4o-mini", "temperature": 0.7, "label": "GPT-4o-mini (T=0.7)"},
    {"model": "gpt-4o-mini", "temperature": 1.0, "label": "GPT-4o-mini (T=1.0)"},
]

results = []
for config in configurations:
    for prompt in test_prompts:
        start_time = time.time()
        response = client.chat.completions.create(
            model=config["model"],
            temperature=config["temperature"],
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300
        )
        elapsed = time.time() - start_time
        content = response.choices[0].message.content
        results.append({
            "config": config["label"],
            "prompt": prompt[:50] + "...",
            "response_length": len(content),
            "latency_ms": round(elapsed * 1000),
            "tokens_used": response.usage.total_tokens
        })

import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

---
## Demo 4: Latency and Cost Analysis

Understanding the cost and latency implications of different models is critical for production deployments. This demo provides utility functions for cost estimation and performance analysis.

In [ ]:
# Cost estimation function
PRICING = {
    "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
    "gpt-4o": {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000},
}

def estimate_cost(model, input_tokens, output_tokens):
    if model in PRICING:
        input_cost = input_tokens * PRICING[model]["input"]
        output_cost = output_tokens * PRICING[model]["output"]
        return round(input_cost + output_cost, 6)
    return None

def analyze_performance(results_df):
    summary = results_df.groupby("config").agg({
        "latency_ms": ["mean", "min", "max"],
        "response_length": "mean",
        "tokens_used": "sum"
    }).round(2)
    print("Performance Summary:")
    print(summary)
    return summary

if len(results) > 0:
    analyze_performance(df)
    
    # Cost estimation example
    for model_name, prices in PRICING.items():
        cost = estimate_cost(model_name, 500, 300)
        print(f"\n{model_name}: Estimated cost for 500 input + 300 output tokens = ${cost}")

---
## Demo 5: A/B Testing Framework

A/B testing lets us rigorously compare two model configurations by running them side-by-side on the same prompts and evaluating both with the LLM judge.

In [ ]:
import random

class ABTestFramework:
    def __init__(self, config_a, config_b):
        self.config_a = config_a
        self.config_b = config_b
        self.results_a = []
        self.results_b = []
        self.client = openai.OpenAI()
    
    def run_test(self, prompts, judge_criteria):
        for prompt in prompts:
            # Get responses from both configurations
            resp_a = self.client.chat.completions.create(
                model=self.config_a["model"],
                temperature=self.config_a.get("temperature", 0.7),
                messages=[
                    {"role": "system", "content": self.config_a.get("system", "You are a helpful assistant.")},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=300
            )
            resp_b = self.client.chat.completions.create(
                model=self.config_b["model"],
                temperature=self.config_b.get("temperature", 0.7),
                messages=[
                    {"role": "system", "content": self.config_b.get("system", "You are a helpful assistant.")},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=300
            )
            
            score_a = llm_judge(prompt, resp_a.choices[0].message.content, judge_criteria)
            score_b = llm_judge(prompt, resp_b.choices[0].message.content, judge_criteria)
            self.results_a.append(score_a)
            self.results_b.append(score_b)
            print(f"Prompt: {prompt[:50]}... | Config A vs B evaluated")
    
    def get_summary(self):
        print(f"\nA/B Test Results:")
        print(f"Config A: {self.config_a.get('label', 'A')}")
        print(f"Config B: {self.config_b.get('label', 'B')}")
        print(f"Tests run: {len(self.results_a)}")

# Demo (small test)
ab_test = ABTestFramework(
    config_a={"model": "gpt-4o-mini", "temperature": 0.0, "label": "Deterministic"},
    config_b={"model": "gpt-4o-mini", "temperature": 0.7, "label": "Creative"}
)
print("A/B Testing Framework initialized!")
print("Run ab_test.run_test(prompts, criteria) to execute tests")

---
## Task 1: Create a Custom Evaluation Rubric for Agentic Tasks

Create a specialized evaluation rubric tailored for agentic AI responses, covering behaviors like reasoning, tool usage, and error handling.

In [ ]:
# TODO: Create a specialized evaluation rubric for agentic AI responses that includes:
# 1. At least 5 criteria specific to agentic behaviors
#    (e.g., tool_usage_accuracy, reasoning_quality, task_decomposition, safety, goal_completion)
# 2. A scoring scale (1-5) with descriptions for each level
# 3. A function that displays the rubric in a formatted way
#
# Hint: Think about what makes an AGENT response different from a regular chatbot response
# Hint: Agents need to plan, use tools, handle errors — evaluate these behaviors
# Hint: Use a nested dictionary structure: {criterion: {score: description}}

def create_agentic_rubric():
    # YOUR CODE HERE
    pass

# Display the rubric
# create_agentic_rubric()

---
## Task 2: Compare Multiple Models on Standardized Tasks

Build a function that systematically compares model configurations by running prompts, scoring with LLM-as-a-Judge, and organizing results into a DataFrame.

In [ ]:
# TODO: Create a model comparison function that:
# 1. Takes a list of test prompts and model configurations
# 2. Runs each prompt through each model configuration
# 3. Evaluates responses using llm_judge (from Demo 2)
# 4. Returns a pandas DataFrame with all scores organized by model and criteria
#
# Hint: Use nested loops — outer for configs, inner for prompts
# Hint: Store results as a list of dicts, then convert to DataFrame
# Hint: Include latency and token count alongside quality scores

def compare_models(prompts, configs, criteria):
    # YOUR CODE HERE
    pass

# test_prompts = ["Explain microservices", "What is CI/CD?"]
# compare_models(test_prompts, configurations, evaluation_criteria)

---
## Task 3: Build an Automated Evaluation Pipeline

Create an end-to-end pipeline that takes test cases, runs them through a model, evaluates with LLM judge, and generates a summary report identifying weak areas.

In [ ]:
# TODO: Build an EvaluationPipeline class that:
# 1. Takes a dataset of (question, expected_behavior) pairs
# 2. Runs the model on each question
# 3. Evaluates using LLM-as-a-judge
# 4. Aggregates scores and identifies weak areas
# 5. Generates a summary report
#
# Hint: Store results in a list, then use pandas for aggregation
# Hint: Weak areas = criteria with average score below 3.5
# Hint: The report should include per-criterion averages and overall score

class EvaluationPipeline:
    def __init__(self, model_config, criteria):
        # YOUR CODE HERE
        pass
    
    def run(self, test_cases):
        # YOUR CODE HERE
        pass
    
    def generate_report(self):
        # YOUR CODE HERE
        pass

---
## Task 4: Analyze and Visualize Evaluation Results

Create visualization functions to plot bar charts, radar charts, and latency-vs-quality scatter plots for comparing evaluation results across models.

In [ ]:
# TODO: Create visualization functions that:
# 1. Plot a bar chart comparing average scores across criteria
# 2. Plot a radar/spider chart for multi-dimensional comparison
# 3. Plot latency vs quality scatter plot
# 4. Generate a summary markdown table
#
# Hint: Use matplotlib for plotting
# Hint: For radar charts, use polar coordinates: ax = plt.subplot(111, polar=True)
# Hint: Add labels, titles, and legends for clarity

import matplotlib.pyplot as plt
import numpy as np

def plot_criteria_comparison(results_df):
    # YOUR CODE HERE
    pass

def plot_radar_chart(scores_dict, title="Model Comparison"):
    # YOUR CODE HERE
    pass

def plot_latency_vs_quality(results_df):
    # YOUR CODE HERE
    pass

# Generate visualizations with sample data or results from Task 2

---
## Summary

In this session, we covered:

1. **Evaluation Rubrics** - Defining structured criteria and scoring scales for consistent assessment
2. **LLM-as-a-Judge** - Using AI to automate evaluation with structured JSON scoring
3. **Benchmarking** - Comparing model configurations across standardized test prompts
4. **Cost and Latency Analysis** - Understanding the practical trade-offs of different models
5. **A/B Testing** - Rigorous side-by-side comparison of model configurations

### Key Takeaways
- Always define clear evaluation criteria before comparing models
- Automated evaluation with LLM judges scales better than manual review
- Consider both quality AND cost/latency when selecting a model for production
- A/B testing provides statistical confidence in model selection decisions
- Visualization helps communicate evaluation results to stakeholders